In [1]:
import copy
from itertools import product
from collections import Counter

import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.visualization import plot_histogram
from scipy.optimize import minimize
import networkx as nx
import matplotlib.pyplot as plt
from numpy import random
from scipy.optimize import minimize
from collections import namedtuple

In [2]:
ideal_simulator = AerSimulator()
sampler = Sampler(mode=ideal_simulator)
n = 10 #number of nodes/qubits

In [3]:
def create_qaoa(graph, num_layers=1):

    #insert edges and nodes
    pqc=QuantumCircuit(n)
    # QAOA circuits start with 
    pqc.h(range(n))

    h = nx.get_node_attributes(graph, "weight")
    J = nx.get_edge_attributes(graph, "weight")
    params={}

    # adding layers 
    for l in range(num_layers):
        
        gamma_label=f'g_{l}'     
        params[gamma_label]= Parameter(gamma_label)    
        
        beta_label=f'b_{l}' 
        params[beta_label]= Parameter(beta_label)    

        pqc.barrier()
        for i in h.keys():
            pqc.rz(h[i]*params[gamma_label], i)
        
        for i, j in J.keys():                
            pqc.cx(i, j)
            pqc.rz(J[i,j]*params[gamma_label], j)
            pqc.cx(i, j)

        #Mixer Hamiltonian
        pqc.barrier()
        for i in range(n):
            pqc.rx(params[beta_label], i)
        
        #end of the layer
        pqc.barrier()

    pqc.measure_all()
    return pqc

def init_params_uni(circuit):
    parameters = random.uniform(-np.pi / 2, np.pi / 2, circuit.num_parameters)
    return parameters

def init_params_lin(circuit):
    parameters = np.linspace(-np.pi / 2, np.pi / 2, circuit.num_parameters)
    return parameters

def init_params_tri(circuit):
    parameters = random.triangular(-np.pi / 2, 0.0, np.pi / 2, circuit.num_parameters)
    return parameters

param_funcs = [init_params_lin, init_params_tri, init_params_uni]

Generate graphs:

In [4]:
def assign_uniform_weights(graph):
    for u, v in graph.edges():
        graph[u][v]["weight"] = random.uniform(-1,1)
    for node in graph.nodes():
        graph.nodes[node]["weight"] = random.uniform(-1,1)
    return graph

In [5]:
def assign_random_weights(graph):
    seq = [-1,1]
    for u, v in graph.edges():
        graph[u][v]["weight"] = random.choice(seq)
    for node in graph.nodes():
        graph.nodes[node]["weight"] = random.choice(seq)
    return graph

In [6]:
def assign_gauss_weights(graph):
    for u, v in graph.edges():
        graph[u][v]["weight"] = random.normal(loc=0.0, scale=1.0)
    for node in graph.nodes():
        graph.nodes[node]["weight"] = random.normal(loc=0.0, scale=1.0)
    return graph

In [7]:
seed = 1224
random.seed(seed)

three_regular_g = nx.random_regular_graph(d=3, n=n, seed=seed)
erdos_renyi_g = nx.erdos_renyi_graph(n=n, p=0.5, seed=seed)
full_g = nx.complete_graph(n)
barabasi_albert_g = nx.barabasi_albert_graph(n=n, m=1, seed=seed)

In [8]:
random_tr_g = assign_random_weights(copy.deepcopy(three_regular_g))
uniform_tr_g = assign_uniform_weights(copy.deepcopy(three_regular_g))
gauss_tr_g = assign_gauss_weights(copy.deepcopy(three_regular_g))

random_er_g = assign_random_weights(copy.deepcopy(erdos_renyi_g))
uniform_er_g = assign_uniform_weights(copy.deepcopy(erdos_renyi_g))
gauss_er_g = assign_gauss_weights(copy.deepcopy(erdos_renyi_g))

random_fc_g = assign_random_weights(copy.deepcopy(full_g))
uniform_fc_g = assign_uniform_weights(copy.deepcopy(full_g))
gauss_fc_g = assign_gauss_weights(copy.deepcopy(full_g))

random_ba_g = assign_random_weights(copy.deepcopy(barabasi_albert_g))
uniform_ba_g = assign_uniform_weights(copy.deepcopy(barabasi_albert_g))
gauss_ba_g = assign_gauss_weights(copy.deepcopy(barabasi_albert_g))

all_graphs = [random_tr_g, uniform_tr_g, gauss_tr_g, random_er_g, uniform_er_g, gauss_er_g, random_fc_g, uniform_fc_g, gauss_fc_g, random_ba_g, uniform_ba_g, gauss_ba_g]

In [9]:
def f(bits, h, J):    
    bitstring = [1 if bit == "0" else -1 for bit in bits]
    return (sum(h[i] * bitstring[i] for i in h) + sum(weight * bitstring[i] * bitstring[j] for (i,j), weight in J.items()))

Run the simulator

In [10]:
def objective(angles, circuit, h, J, shots=1024):
    circuit_with_numeric_angles = circuit.assign_parameters(angles)
    run = sampler.run([circuit_with_numeric_angles], shots=shots)
    ideal_counts = run.result()[0].data.meas.get_counts()

    EV = 0.0
    for bitstring, count in ideal_counts.items():
        cost = f(bitstring[::-1], h, J)     # f is your cost function
        EV += cost * count

    EV /= sum(ideal_counts.values())
    return EV

In [11]:
import os
import pickle
from itertools import product
import numpy as np
import networkx as nx
from scipy.optimize import minimize
from qiskit import transpile
from qiskit_ibm_runtime.fake_provider import FakeFez      # pick any fake device with enough qubits
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.primitives import StatevectorSampler

results_path = "qaoa_ideal_results.pkl"
max_layers = 10
shots = 1024

sampler = StatevectorSampler(seed=1224)       # ideal: exact statevector, shot noise only

# Load earlier progress if the file exists
if os.path.exists(results_path):
    with open(results_path, "rb") as file:
        all_results = pickle.load(file)
else:
    all_results = []

completed_run_keys = {
    (r["graph_index"], r["num_layers"], r["init_name"]) for r in all_results
}


def save_results():
    temporary_path = results_path + ".tmp"
    with open(temporary_path, "wb") as file:
        pickle.dump(all_results, file)
    os.replace(temporary_path, results_path)     # swap in only after the write finished


for graph_index, graph in enumerate(all_graphs):
    h = nx.get_node_attributes(graph, "weight")
    J = nx.get_edge_attributes(graph, "weight")
    n = graph.number_of_nodes()

    all_bitstrings = ["".join(b) for b in product("01", repeat=n)]
    all_costs = [f(bits, h, J) for bits in all_bitstrings]
    exact_global_minimum = min(all_costs)
    exact_best_bitstring = all_bitstrings[int(np.argmin(all_costs))]

    for num_layers in range(1, max_layers + 1):
        if all((graph_index, num_layers, p.__name__) in completed_run_keys for p in param_funcs):
            continue                                       # whole depth already done

        pqc = create_qaoa(graph=graph, num_layers=num_layers)

        # Circuit statistics: same for every initializer at this graph and depth
        circuit_stats = {
            "logical_depth": pqc.depth(),
            "logical_two_qubit_gates": pqc.num_nonlocal_gates(),
            "logical_gate_counts": dict(pqc.count_ops()),
        }

        for init_param in param_funcs:
            run_key = (graph_index, num_layers, init_param.__name__)
            if run_key in completed_run_keys:
                continue                                   # finished in an earlier session

            print(
                f"Simulating (ideal) graph {graph_index + 1}/{len(all_graphs)} | "
                f"layers {num_layers}/{max_layers} | init {init_param.__name__} | "
                f"{n} qubits, {pqc.num_parameters} parameters, depth {pqc.depth()}"
            )

            x0 = init_param(pqc)
            evaluations = []

            def logged_objective(angles):
                counts = (
                    sampler.run([pqc.assign_parameters(angles)], shots=shots)
                    .result()[0].data.meas.get_counts()
                )
                counts = {b[::-1]: c for b, c in counts.items()}
                ev = sum(f(b, h, J) * c for b, c in counts.items()) / sum(counts.values())
                evaluations.append({"angles": np.array(angles), "ev": ev, "counts": counts})
                return ev

            result = minimize(logged_objective, x0, method="COBYLA", options={"maxiter": 200})
            print(f"  done: EV {result.fun:.4f} vs exact {exact_global_minimum:.4f}, {result.nfev} evaluations")

            all_results.append({
                "graph_index": graph_index,
                "num_layers": num_layers,
                "init_name": init_param.__name__,
                "parameter_names": [p.name for p in pqc.parameters],
                "x0": x0,
                "result": result,
                "evaluations": evaluations,
                "all_bitstrings": all_bitstrings,
                "all_costs": all_costs,
                "exact_global_minimum": exact_global_minimum,
                "exact_best_bitstring": exact_best_bitstring,
                **circuit_stats,
            })
            completed_run_keys.add(run_key)
            save_results()                                 # checkpoint after every run

In [12]:
# Create a noisy simulator from the selected IBM backend
fake_device = FakeFez()                           # local object holding the device's calibration data
noisy_backend = AerSimulator.from_backend(fake_device)   # local simulator with that device's noise model, basis gates and coupling map

sampler = SamplerV2.from_backend(noisy_backend, seed=1224)
pass_manager = generate_preset_pass_manager(optimization_level=1, backend=noisy_backend)

In [13]:
results_path = "qaoa_noisy_results.pkl"

# Load earlier progress if the file exists
if os.path.exists(results_path):
    with open(results_path, "rb") as file:
        all_results = pickle.load(file)
else:
    all_results = []

completed_run_keys = {
    (r["graph_index"], r["num_layers"], r["init_name"]) for r in all_results
}


def save_results():
    temporary_path = results_path + ".tmp"
    with open(temporary_path, "wb") as file:
        pickle.dump(all_results, file)
    os.replace(temporary_path, results_path)     # swap in only after the write finished


for graph_index, graph in enumerate(all_graphs):
    h = nx.get_node_attributes(graph, "weight")
    J = nx.get_edge_attributes(graph, "weight")
    n = graph.number_of_nodes()

    all_bitstrings = ["".join(b) for b in product("01", repeat=n)]
    all_costs = [f(bits, h, J) for bits in all_bitstrings]
    exact_global_minimum = min(all_costs)
    exact_best_bitstring = all_bitstrings[int(np.argmin(all_costs))]

    for num_layers in range(1, 11):
        logical_pqc = create_qaoa(graph=graph, num_layers=num_layers)
        pqc = pass_manager.run(logical_pqc)

        circuit_stats = {
            "logical_depth": logical_pqc.depth(),
            "logical_two_qubit_gates": logical_pqc.num_nonlocal_gates(),
            "transpiled_depth": pqc.depth(),
            "transpiled_two_qubit_gates": pqc.num_nonlocal_gates(),
            "transpiled_gate_counts": dict(pqc.count_ops()),
        }

        for init_param in param_funcs:
            run_key = (graph_index, num_layers, init_param.__name__)
            if run_key in completed_run_keys:
                continue                                   # finished in an earlier session
            print(
                f"Simulating graph {graph_index + 1}/{len(all_graphs)} | "
                f"layers {num_layers}/5 | init {init_param.__name__} | "
                f"{n} qubits, {pqc.num_parameters} parameters, depth {pqc.depth()}"
            )


            x0 = init_param(pqc)
            evaluations = []

            def logged_objective(angles):
                counts = (
                    sampler.run([pqc.assign_parameters(angles)], shots=1024)
                    .result()[0].data.meas.get_counts()
                )
                counts = {b[::-1]: c for b, c in counts.items()}
                ev = sum(f(b, h, J) * c for b, c in counts.items()) / sum(counts.values())
                evaluations.append({"angles": np.array(angles), "ev": ev, "counts": counts})
                return ev

            result = minimize(logged_objective, x0, method="COBYLA", options={"maxiter": 200})

            all_results.append({
                "graph_index": graph_index,
                "num_layers": num_layers,
                "init_name": init_param.__name__,
                "parameter_names": [p.name for p in pqc.parameters],
                "x0": x0,
                "result": result,
                "evaluations": evaluations,
                "all_bitstrings": all_bitstrings,
                "all_costs": all_costs,
                "exact_global_minimum": exact_global_minimum,
                "exact_best_bitstring": exact_best_bitstring,
                **circuit_stats
            })
            completed_run_keys.add(run_key)
            save_results()                                 # checkpoint after every run

Simulating graph 9/12 | layers 9/5 | init init_params_tri | 10 qubits, 18 parameters, depth 3360
Simulating graph 9/12 | layers 9/5 | init init_params_uni | 10 qubits, 18 parameters, depth 3360
Simulating graph 9/12 | layers 10/5 | init init_params_lin | 10 qubits, 20 parameters, depth 3649
Simulating graph 9/12 | layers 10/5 | init init_params_tri | 10 qubits, 20 parameters, depth 3649
Simulating graph 9/12 | layers 10/5 | init init_params_uni | 10 qubits, 20 parameters, depth 3649
Simulating graph 10/12 | layers 8/5 | init init_params_lin | 10 qubits, 16 parameters, depth 725
Simulating graph 10/12 | layers 8/5 | init init_params_tri | 10 qubits, 16 parameters, depth 725
Simulating graph 10/12 | layers 8/5 | init init_params_uni | 10 qubits, 16 parameters, depth 725
Simulating graph 10/12 | layers 9/5 | init init_params_lin | 10 qubits, 18 parameters, depth 810
Simulating graph 10/12 | layers 9/5 | init init_params_tri | 10 qubits, 18 parameters, depth 810
Simulating graph 10/12 | la

In [14]:
import numpy as np

target = fake_device.target
skipped_operations = {"delay"}


def sci(value):
    return "--" if value is None else f"{value:.2e}"


def fixed(value, scale, digits=1):
    return "--" if value is None else f"{value * scale:.{digits}f}"


def escape(text):
    return str(text).replace("_", r"\_")


def mean_of(values):
    values = [v for v in values if v is not None]
    return float(np.mean(values)) if values else None


def latex_table(header, rows, caption, label, long=True):
    column_format = "l" + "r" * (len(header) - 1)
    header_line = " & ".join(escape(h) for h in header) + r" \\"
    body = [" & ".join(row) + r" \\" for row in rows]

    if long:
        lines = [
            r"{\footnotesize\setlength{\tabcolsep}{4pt}",
            r"\begin{longtable}{" + column_format + "}",
            rf"\caption{{{caption}}} \label{{{label}}} \\",
            r"\toprule", header_line, r"\midrule", r"\endfirsthead",
            r"\toprule", header_line, r"\midrule", r"\endhead",
            *body,
            r"\bottomrule", r"\end{longtable}}",
        ]
    else:
        lines = [
            r"\begin{table}[htbp]", r"\centering", r"\small",
            r"\begin{tabular}{" + column_format + "}",
            r"\toprule", header_line, r"\midrule",
            *body,
            r"\bottomrule", r"\end{tabular}",
            rf"\caption{{{caption}}}", rf"\label{{{label}}}", r"\end{table}",
        ]
    return "\n".join(lines)


def qubit_arguments(gate_name):
    return [q for q in target[gate_name] if q is not None]


one_qubit_gates = [g for g in target.operation_names if g not in skipped_operations | {"measure"}
                   and qubit_arguments(g) and all(len(q) == 1 for q in qubit_arguments(g))]
two_qubit_gates = [g for g in target.operation_names if g not in skipped_operations
                   and qubit_arguments(g) and all(len(q) == 2 for q in qubit_arguments(g))]

# ================= SHORT SUMMARY TABLES =================

# Averages per gate type (readout is the "measure" instruction)
rows = []
for gate in one_qubit_gates + two_qubit_gates + ["measure"]:
    properties = [p for p in target[gate].values() if p is not None]
    name = "measure (readout)" if gate == "measure" else gate
    rows.append([escape(name),
                 sci(mean_of([p.error for p in properties])),
                 fixed(mean_of([p.duration for p in properties]), 1e9)])
print(latex_table(["Operation", "Mean error", "Mean duration (ns)"], rows,
                  f"Average gate error and duration on {escape(fake_device.name)}.",
                  "tab:gate-averages", long=False))

# Averages of qubit coherence
qubit_properties = [target.qubit_properties[q] for q in range(target.num_qubits)]
rows = [
    ["T1 (µs)", fixed(mean_of([p.t1 for p in qubit_properties]), 1e6)],
    ["T2 (µs)", fixed(mean_of([p.t2 for p in qubit_properties]), 1e6)],
]
print(latex_table(["Property", "Mean"], rows,
                  f"Average qubit coherence times on {escape(fake_device.name)}.",
                  "tab:coherence-averages", long=False))

# ================= DETAILED TABLES, ONE PER GATE =================

# Per-qubit properties: three columns, no readout duration
rows = []
for qubit in range(target.num_qubits):
    properties = qubit_properties[qubit]
    readout = target["measure"][(qubit,)]
    rows.append([str(qubit), fixed(properties.t1, 1e6), fixed(properties.t2, 1e6), sci(readout.error)])
print(latex_table(["Qubit", "T1 (µs)", "T2 (µs)", "Readout error"], rows,
                  "Per-qubit coherence and readout error.", "tab:qubit-properties"))

# One table per single-qubit gate
for gate in one_qubit_gates:
    rows = []
    for qubit in range(target.num_qubits):
        properties = target[gate].get((qubit,))
        rows.append([str(qubit),
                     sci(properties.error if properties else None),
                     fixed(properties.duration if properties else None, 1e9)])
    print(latex_table(["Qubit", "Error", "Duration (ns)"], rows,
                      f"Error and duration of the {escape(gate)} gate per qubit.", f"tab:{gate}-gate"))

# One table per two-qubit gate
for gate in two_qubit_gates:
    rows = []
    for pair in sorted(qubit_arguments(gate)):
        properties = target[gate][pair]
        rows.append([f"({pair[0]}, {pair[1]})", sci(properties.error), fixed(properties.duration, 1e9)])
    print(latex_table(["Qubits", "Error", "Duration (ns)"], rows,
                      f"Error and duration of the {escape(gate)} gate per qubit pair.", f"tab:{gate}-gate"))

\begin{table}[htbp]
\centering
\small
\begin{tabular}{lrr}
\toprule
Operation & Mean error & Mean duration (ns) \\
\midrule
rz & 0.00e+00 & 0.0 \\
reset & -- & 1584.0 \\
sx & 2.87e-04 & 24.0 \\
x & 2.87e-04 & 24.0 \\
id & 2.87e-04 & 24.0 \\
cz & 4.51e-02 & 84.2 \\
measure (readout) & 1.32e-02 & 1560.0 \\
\bottomrule
\end{tabular}
\caption{Average gate error and duration on fake\_fez.}
\label{tab:gate-averages}
\end{table}
\begin{table}[htbp]
\centering
\small
\begin{tabular}{lr}
\toprule
Property & Mean \\
\midrule
T1 (µs) & 145.3 \\
T2 (µs) & 90.5 \\
\bottomrule
\end{tabular}
\caption{Average qubit coherence times on fake\_fez.}
\label{tab:coherence-averages}
\end{table}
{\footnotesize\setlength{\tabcolsep}{4pt}
\begin{longtable}{lrrr}
\caption{Per-qubit coherence and readout error.} \label{tab:qubit-properties} \\
\toprule
Qubit & T1 (µs) & T2 (µs) & Readout error \\
\midrule
\endfirsthead
\toprule
Qubit & T1 (µs) & T2 (µs) & Readout error \\
\midrule
\endhead
0 & 48.8 & 42.4 & 1.15e

In [15]:
# fez_connectivity = nx.Graph()
# fez_connectivity.add_edges_from(fake_device.coupling_map)


# import math

# # graph = your instantiated networkx graph
# print(nx.check_planarity(fez_connectivity)[0])           # True means a crossing-free drawing exists

# start_positions = nx.planar_layout(fez_connectivity)
# node_positions = nx.kamada_kawai_layout(fez_connectivity, pos=start_positions, scale=3.0)

# fig, ax = plt.subplots(figsize=(16, 16))
# nx.draw(
#     fez_connectivity, node_positions, ax=ax,
#     with_labels=True, node_size=250, font_size=7,
#     node_color="lightsteelblue", edge_color="gray",
# )
# ax.margins(0.05)
# plt.show()